> Note: This notebook is an exercise workbook companion. Some cells intentionally contain `TODO` prompts or `NotImplementedError` placeholders for the reader to complete. For fully maintained runnable reference code, see `src/`, `tests/`, and the printed listings in the book.

# CAPSTONE PROJECT: The Optimizer Showdown

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/your-repo/Optimization-for-Machine-Learning/blob/main/notebooks/capstone_optimizer_showdown.ipynb)

---

## The Ultimate Optimizer Benchmark

**Welcome to the grand finale!** In this capstone project, we'll implement the most important optimizers from scratch, train neural networks on MNIST, and conduct a comprehensive benchmark to determine the best optimizer for deep learning.

---

## Learning Objectives

By completing this project, you will:

1. **Implement optimizers from scratch**: SGD, Momentum, RMSprop, Adam
2. **Build an MLP** from first principles using only NumPy
3. **Train on MNIST** and achieve >95% accuracy
4. **Benchmark comprehensively**: convergence speed, stability, hyperparameter sensitivity
5. **Visualize and compare** optimizer behaviors
6. **Draw conclusions** for practical deep learning

---

## Table of Contents

1. [Setup and Data Loading](#1-setup)
2. [MLP Implementation from Scratch](#2-mlp)
3. [Optimizer Implementations](#3-optimizers)
4. [Training Infrastructure](#4-training)
5. [Benchmark Suite](#5-benchmark)
6. [Results and Analysis](#6-results)
7. [Conclusions and Recommendations](#7-conclusions)

---

## 1. Setup and Data Loading <a name="1-setup"></a>

We'll use only NumPy for the implementations (no PyTorch/TensorFlow for the core logic).
We'll load MNIST using scikit-learn or fetch it manually.

In [ ]:
# Core imports
import numpy as np
import matplotlib.pyplot as plt
from typing import Dict, List, Tuple, Callable
import time
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

# For data loading
try:
    from sklearn.datasets import fetch_openml
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import StandardScaler
    SKLEARN_AVAILABLE = True
except ImportError:
    SKLEARN_AVAILABLE = False
    print("scikit-learn not found. Will use synthetic data.")

# Random seed for reproducibility
np.random.seed(42)

# Plotting configuration
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.labelsize'] = 14
plt.rcParams['axes.titlesize'] = 16
plt.rcParams['legend.fontsize'] = 11

# Color palette
COLORS = {
    'sgd': '#2E86AB',      # Blue
    'momentum': '#A23B72', # Magenta
    'rmsprop': '#F18F01',  # Orange
    'adam': '#C73E1D',     # Red
    'training': '#3B7A57', # Green
    'validation': '#6B4E71'  # Purple
}

print("Setup complete!")
print(f"NumPy version: {np.__version__}")

In [ ]:
def load_mnist_data():
    """
    Load and preprocess MNIST dataset.
    
    Returns:
    --------
    X_train, X_val, X_test : normalized image data
    y_train, y_val, y_test : one-hot encoded labels
    """
    print("Loading MNIST dataset...")
    
    if SKLEARN_AVAILABLE:
        # Load MNIST
        mnist = fetch_openml('mnist_784', version=1, as_frame=False, parser='auto')
        X, y = mnist.data.astype(np.float32), mnist.target.astype(np.int32)
        
        # Normalize to [0, 1]
        X = X / 255.0
        
        # Split: 50000 train, 10000 val, 10000 test
        X_train, X_temp, y_train, y_temp = train_test_split(
            X, y, test_size=20000, random_state=42, stratify=y
        )
        X_val, X_test, y_val, y_test = train_test_split(
            X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
        )
    else:
        # Create synthetic data for testing
        print("Using synthetic data (install sklearn for real MNIST)")
        n_train, n_val, n_test = 5000, 1000, 1000
        n_features = 784
        n_classes = 10
        
        X_train = np.random.randn(n_train, n_features).astype(np.float32)
        X_val = np.random.randn(n_val, n_features).astype(np.float32)
        X_test = np.random.randn(n_test, n_features).astype(np.float32)
        
        y_train = np.random.randint(0, n_classes, n_train)
        y_val = np.random.randint(0, n_classes, n_val)
        y_test = np.random.randint(0, n_classes, n_test)
    
    # One-hot encode labels
    def one_hot(y, n_classes=10):
        return np.eye(n_classes)[y]
    
    y_train_oh = one_hot(y_train)
    y_val_oh = one_hot(y_val)
    y_test_oh = one_hot(y_test)
    
    print(f"\nDataset shapes:")
    print(f"  Train: X={X_train.shape}, y={y_train_oh.shape}")
    print(f"  Val:   X={X_val.shape}, y={y_val_oh.shape}")
    print(f"  Test:  X={X_test.shape}, y={y_test_oh.shape}")
    
    return (X_train, y_train, y_train_oh), (X_val, y_val, y_val_oh), (X_test, y_test, y_test_oh)

# Load data
train_data, val_data, test_data = load_mnist_data()
X_train, y_train, y_train_oh = train_data
X_val, y_val, y_val_oh = val_data
X_test, y_test, y_test_oh = test_data

In [ ]:
# Visualize some samples

fig, axes = plt.subplots(2, 10, figsize=(15, 4))

for i in range(20):
    ax = axes[i // 10, i % 10]
    ax.imshow(X_train[i].reshape(28, 28), cmap='gray')
    ax.set_title(f'{y_train[i]}', fontsize=10)
    ax.axis('off')

plt.suptitle('MNIST Sample Images', fontsize=14)
plt.tight_layout()
plt.savefig('mnist_samples.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. MLP Implementation from Scratch <a name="2-mlp"></a>

We'll implement a Multi-Layer Perceptron (MLP) from scratch with:
- Flexible architecture (any number of layers)
- ReLU and Softmax activations
- Xavier/He weight initialization
- Cross-entropy loss
- Mini-batch training support

In [ ]:
class MLP:
    """
    Multi-Layer Perceptron implemented from scratch.
    
    Architecture: Input -> [Hidden + ReLU]* -> Output + Softmax
    
    Parameters:
    -----------
    layer_sizes : list
        List of layer sizes, e.g., [784, 256, 128, 10]
    """
    
    def __init__(self, layer_sizes: List[int], seed: int = 42):
        """Initialize the network with Xavier/He initialization."""
        np.random.seed(seed)
        self.layer_sizes = layer_sizes
        self.n_layers = len(layer_sizes) - 1
        
        # Initialize weights and biases
        self.weights = []
        self.biases = []
        
        for i in range(self.n_layers):
            # He initialization for ReLU layers
            fan_in = layer_sizes[i]
            fan_out = layer_sizes[i + 1]
            
            # He initialization: std = sqrt(2 / fan_in)
            std = np.sqrt(2.0 / fan_in)
            W = np.random.randn(fan_in, fan_out).astype(np.float32) * std
            b = np.zeros(fan_out, dtype=np.float32)
            
            self.weights.append(W)
            self.biases.append(b)
        
        # Cache for backprop
        self.cache = {}
    
    def relu(self, x):
        """ReLU activation."""
        return np.maximum(0, x)
    
    def relu_derivative(self, x):
        """ReLU derivative."""
        return (x > 0).astype(np.float32)
    
    def softmax(self, x):
        """Numerically stable softmax."""
        # Subtract max for numerical stability
        exp_x = np.exp(x - np.max(x, axis=1, keepdims=True))
        return exp_x / np.sum(exp_x, axis=1, keepdims=True)
    
    def forward(self, X: np.ndarray, training: bool = True) -> np.ndarray:
        """
        Forward pass through the network.
        
        Parameters:
        -----------
        X : np.ndarray
            Input data of shape (batch_size, input_dim)
        training : bool
            If True, cache activations for backprop
        
        Returns:
        --------
        output : np.ndarray
            Network output (probabilities) of shape (batch_size, n_classes)
        """
        if training:
            self.cache = {'activations': [X], 'pre_activations': []}
        
        current = X
        
        for i in range(self.n_layers):
            # Linear transformation
            z = current @ self.weights[i] + self.biases[i]
            
            if training:
                self.cache['pre_activations'].append(z)
            
            # Activation
            if i < self.n_layers - 1:
                # Hidden layers: ReLU
                current = self.relu(z)
            else:
                # Output layer: Softmax
                current = self.softmax(z)
            
            if training:
                self.cache['activations'].append(current)
        
        return current
    
    def backward(self, y_true: np.ndarray) -> Tuple[List[np.ndarray], List[np.ndarray]]:
        """
        Backward pass to compute gradients.
        
        Parameters:
        -----------
        y_true : np.ndarray
            One-hot encoded true labels of shape (batch_size, n_classes)
        
        Returns:
        --------
        weight_grads : list of np.ndarray
            Gradients for weights
        bias_grads : list of np.ndarray
            Gradients for biases
        """
        batch_size = y_true.shape[0]
        
        weight_grads = [None] * self.n_layers
        bias_grads = [None] * self.n_layers
        
        # Output layer: softmax + cross-entropy gradient is (y_pred - y_true)
        delta = self.cache['activations'][-1] - y_true  # (batch_size, n_classes)
        
        # Backpropagate through layers
        for i in range(self.n_layers - 1, -1, -1):
            # Gradient w.r.t. weights and biases
            activation_prev = self.cache['activations'][i]
            weight_grads[i] = (activation_prev.T @ delta) / batch_size
            bias_grads[i] = np.mean(delta, axis=0)
            
            if i > 0:
                # Backpropagate through ReLU
                delta = delta @ self.weights[i].T
                delta = delta * self.relu_derivative(self.cache['pre_activations'][i-1])
        
        return weight_grads, bias_grads
    
    def get_params(self) -> List[np.ndarray]:
        """Get all parameters as a flat list."""
        params = []
        for w, b in zip(self.weights, self.biases):
            params.extend([w, b])
        return params
    
    def set_params(self, params: List[np.ndarray]):
        """Set all parameters from a flat list."""
        idx = 0
        for i in range(self.n_layers):
            self.weights[i] = params[idx]
            self.biases[i] = params[idx + 1]
            idx += 2
    
    def get_grads(self, X: np.ndarray, y: np.ndarray) -> List[np.ndarray]:
        """Compute gradients for given batch."""
        self.forward(X, training=True)
        weight_grads, bias_grads = self.backward(y)
        
        grads = []
        for wg, bg in zip(weight_grads, bias_grads):
            grads.extend([wg, bg])
        return grads
    
    def copy(self):
        """Create a copy of the network."""
        new_mlp = MLP(self.layer_sizes)
        new_mlp.weights = [w.copy() for w in self.weights]
        new_mlp.biases = [b.copy() for b in self.biases]
        return new_mlp

print("MLP class defined!")
print(f"\nTest forward pass:")
test_mlp = MLP([784, 256, 128, 10])
test_output = test_mlp.forward(X_train[:32])
print(f"  Input shape: (32, 784)")
print(f"  Output shape: {test_output.shape}")
print(f"  Output sums to 1: {np.allclose(test_output.sum(axis=1), 1.0)}")

In [ ]:
def cross_entropy_loss(y_pred: np.ndarray, y_true: np.ndarray) -> float:
    """
    Compute cross-entropy loss.
    
    Parameters:
    -----------
    y_pred : np.ndarray
        Predicted probabilities (batch_size, n_classes)
    y_true : np.ndarray
        One-hot encoded true labels (batch_size, n_classes)
    
    Returns:
    --------
    loss : float
        Average cross-entropy loss
    """
    # Clip to avoid log(0)
    eps = 1e-12
    y_pred = np.clip(y_pred, eps, 1 - eps)
    
    # Cross-entropy: -sum(y_true * log(y_pred))
    loss = -np.mean(np.sum(y_true * np.log(y_pred), axis=1))
    return loss


def accuracy(y_pred: np.ndarray, y_true: np.ndarray) -> float:
    """
    Compute classification accuracy.
    
    Parameters:
    -----------
    y_pred : np.ndarray
        Predicted probabilities or logits
    y_true : np.ndarray
        True labels (integer or one-hot)
    
    Returns:
    --------
    acc : float
        Accuracy in [0, 1]
    """
    pred_labels = np.argmax(y_pred, axis=1)
    
    if len(y_true.shape) > 1:
        true_labels = np.argmax(y_true, axis=1)
    else:
        true_labels = y_true
    
    return np.mean(pred_labels == true_labels)


# Test loss and accuracy
test_pred = test_mlp.forward(X_train[:100], training=False)
test_loss = cross_entropy_loss(test_pred, y_train_oh[:100])
test_acc = accuracy(test_pred, y_train[:100])

print(f"Test loss (untrained): {test_loss:.4f}")
print(f"Test accuracy (untrained): {test_acc:.2%} (expected ~10% for random)")

## 3. Optimizer Implementations <a name="3-optimizers"></a>

We'll implement four fundamental optimizers from scratch:

1. **SGD (Stochastic Gradient Descent)**: The baseline
2. **Momentum**: SGD with velocity accumulation
3. **RMSprop**: Adaptive learning rates per parameter
4. **Adam**: Combines momentum and RMSprop

Each optimizer follows the same interface for easy comparison.

In [ ]:
class Optimizer:
    """Base class for optimizers."""
    
    def __init__(self, params: List[np.ndarray], lr: float = 0.01):
        """
        Initialize optimizer.
        
        Parameters:
        -----------
        params : list of np.ndarray
            List of parameter arrays to optimize
        lr : float
            Learning rate
        """
        self.params = params
        self.lr = lr
        self.t = 0  # Time step counter
    
    def step(self, grads: List[np.ndarray]):
        """Update parameters using gradients."""
        raise NotImplementedError
    
    def zero_grad(self):
        """Reset any accumulated gradients (for compatibility)."""
        pass


class SGD(Optimizer):
    """
    Stochastic Gradient Descent.
    
    Update rule:
        theta = theta - lr * gradient
    """
    
    def __init__(self, params: List[np.ndarray], lr: float = 0.01):
        super().__init__(params, lr)
        self.name = 'SGD'
    
    def step(self, grads: List[np.ndarray]):
        """Perform one optimization step."""
        self.t += 1
        for i, grad in enumerate(grads):
            self.params[i] -= self.lr * grad


class MomentumSGD(Optimizer):
    """
    SGD with Momentum.
    
    Update rule:
        v = beta * v + gradient
        theta = theta - lr * v
    
    The momentum term accumulates past gradients, helping to:
    - Accelerate in consistent gradient directions
    - Dampen oscillations in high-curvature directions
    """
    
    def __init__(self, params: List[np.ndarray], lr: float = 0.01, beta: float = 0.9):
        super().__init__(params, lr)
        self.beta = beta
        self.name = f'Momentum(beta={beta})'
        
        # Initialize velocity
        self.v = [np.zeros_like(p) for p in params]
    
    def step(self, grads: List[np.ndarray]):
        """Perform one optimization step with momentum."""
        self.t += 1
        for i, grad in enumerate(grads):
            # Update velocity
            self.v[i] = self.beta * self.v[i] + grad
            # Update parameters
            self.params[i] -= self.lr * self.v[i]


class RMSprop(Optimizer):
    """
    RMSprop: Root Mean Square Propagation.
    
    Update rule:
        s = beta * s + (1 - beta) * gradient^2
        theta = theta - lr * gradient / (sqrt(s) + eps)
    
    Adapts learning rate per-parameter based on recent gradient magnitudes.
    Good for non-stationary objectives and sparse gradients.
    """
    
    def __init__(self, params: List[np.ndarray], lr: float = 0.001, 
                 beta: float = 0.99, eps: float = 1e-8):
        super().__init__(params, lr)
        self.beta = beta
        self.eps = eps
        self.name = f'RMSprop(beta={beta})'
        
        # Initialize squared gradient accumulator
        self.s = [np.zeros_like(p) for p in params]
    
    def step(self, grads: List[np.ndarray]):
        """Perform one optimization step."""
        self.t += 1
        for i, grad in enumerate(grads):
            # Update squared gradient accumulator
            self.s[i] = self.beta * self.s[i] + (1 - self.beta) * (grad ** 2)
            # Update parameters with adaptive learning rate
            self.params[i] -= self.lr * grad / (np.sqrt(self.s[i]) + self.eps)


class Adam(Optimizer):
    """
    Adam: Adaptive Moment Estimation.
    
    Combines momentum (first moment) and RMSprop (second moment).
    
    Update rule:
        m = beta1 * m + (1 - beta1) * gradient          # First moment
        v = beta2 * v + (1 - beta2) * gradient^2        # Second moment
        m_hat = m / (1 - beta1^t)                       # Bias correction
        v_hat = v / (1 - beta2^t)                       # Bias correction
        theta = theta - lr * m_hat / (sqrt(v_hat) + eps)
    
    Default hyperparameters (beta1=0.9, beta2=0.999) work well in practice.
    """
    
    def __init__(self, params: List[np.ndarray], lr: float = 0.001,
                 beta1: float = 0.9, beta2: float = 0.999, eps: float = 1e-8):
        super().__init__(params, lr)
        self.beta1 = beta1
        self.beta2 = beta2
        self.eps = eps
        self.name = f'Adam(b1={beta1},b2={beta2})'
        
        # Initialize moment estimates
        self.m = [np.zeros_like(p) for p in params]  # First moment
        self.v = [np.zeros_like(p) for p in params]  # Second moment
    
    def step(self, grads: List[np.ndarray]):
        """Perform one optimization step."""
        self.t += 1
        
        for i, grad in enumerate(grads):
            # Update biased first moment estimate
            self.m[i] = self.beta1 * self.m[i] + (1 - self.beta1) * grad
            
            # Update biased second moment estimate
            self.v[i] = self.beta2 * self.v[i] + (1 - self.beta2) * (grad ** 2)
            
            # Compute bias-corrected estimates
            m_hat = self.m[i] / (1 - self.beta1 ** self.t)
            v_hat = self.v[i] / (1 - self.beta2 ** self.t)
            
            # Update parameters
            self.params[i] -= self.lr * m_hat / (np.sqrt(v_hat) + self.eps)


print("Optimizers implemented:")
print("  1. SGD")
print("  2. MomentumSGD")
print("  3. RMSprop")
print("  4. Adam")

## 4. Training Infrastructure <a name="4-training"></a>

We'll create a training loop that:
- Supports mini-batch training
- Tracks loss and accuracy over epochs
- Measures wall-clock time
- Provides progress updates

In [ ]:
class Trainer:
    """
    Training infrastructure for neural networks.
    
    Features:
    - Mini-batch training
    - Validation evaluation
    - Loss and accuracy tracking
    - Early stopping (optional)
    - Time tracking
    """
    
    def __init__(self, model: MLP, optimizer: Optimizer):
        """
        Initialize trainer.
        
        Parameters:
        -----------
        model : MLP
            Neural network model
        optimizer : Optimizer
            Optimizer instance
        """
        self.model = model
        self.optimizer = optimizer
        self.history = {
            'train_loss': [],
            'train_acc': [],
            'val_loss': [],
            'val_acc': [],
            'epoch_times': [],
            'total_time': 0
        }
    
    def train_epoch(self, X: np.ndarray, y: np.ndarray, 
                    batch_size: int = 64, shuffle: bool = True) -> Tuple[float, float]:
        """
        Train for one epoch.
        
        Returns:
        --------
        avg_loss : float
        avg_acc : float
        """
        n_samples = X.shape[0]
        
        # Shuffle data
        if shuffle:
            indices = np.random.permutation(n_samples)
            X = X[indices]
            y = y[indices]
        
        # Mini-batch training
        total_loss = 0
        total_correct = 0
        n_batches = 0
        
        for i in range(0, n_samples, batch_size):
            X_batch = X[i:i+batch_size]
            y_batch = y[i:i+batch_size]
            
            # Forward pass
            y_pred = self.model.forward(X_batch, training=True)
            
            # Compute loss
            batch_loss = cross_entropy_loss(y_pred, y_batch)
            total_loss += batch_loss * len(X_batch)
            
            # Compute accuracy
            total_correct += np.sum(np.argmax(y_pred, axis=1) == np.argmax(y_batch, axis=1))
            
            # Backward pass
            weight_grads, bias_grads = self.model.backward(y_batch)
            grads = []
            for wg, bg in zip(weight_grads, bias_grads):
                grads.extend([wg, bg])
            
            # Update parameters
            self.optimizer.step(grads)
            
            n_batches += 1
        
        avg_loss = total_loss / n_samples
        avg_acc = total_correct / n_samples
        
        return avg_loss, avg_acc
    
    def evaluate(self, X: np.ndarray, y: np.ndarray, 
                 batch_size: int = 256) -> Tuple[float, float]:
        """
        Evaluate model on data.
        
        Returns:
        --------
        avg_loss : float
        avg_acc : float
        """
        n_samples = X.shape[0]
        total_loss = 0
        total_correct = 0
        
        for i in range(0, n_samples, batch_size):
            X_batch = X[i:i+batch_size]
            y_batch = y[i:i+batch_size]
            
            y_pred = self.model.forward(X_batch, training=False)
            
            batch_loss = cross_entropy_loss(y_pred, y_batch)
            total_loss += batch_loss * len(X_batch)
            
            total_correct += np.sum(np.argmax(y_pred, axis=1) == np.argmax(y_batch, axis=1))
        
        avg_loss = total_loss / n_samples
        avg_acc = total_correct / n_samples
        
        return avg_loss, avg_acc
    
    def train(self, X_train: np.ndarray, y_train: np.ndarray,
              X_val: np.ndarray, y_val: np.ndarray,
              epochs: int = 20, batch_size: int = 64,
              verbose: bool = True) -> Dict:
        """
        Full training loop.
        
        Parameters:
        -----------
        X_train, y_train : training data
        X_val, y_val : validation data
        epochs : int
            Number of training epochs
        batch_size : int
            Mini-batch size
        verbose : bool
            Print progress
        
        Returns:
        --------
        history : dict
            Training history
        """
        start_time = time.time()
        
        for epoch in range(epochs):
            epoch_start = time.time()
            
            # Train
            train_loss, train_acc = self.train_epoch(X_train, y_train, batch_size)
            
            # Validate
            val_loss, val_acc = self.evaluate(X_val, y_val)
            
            epoch_time = time.time() - epoch_start
            
            # Record history
            self.history['train_loss'].append(train_loss)
            self.history['train_acc'].append(train_acc)
            self.history['val_loss'].append(val_loss)
            self.history['val_acc'].append(val_acc)
            self.history['epoch_times'].append(epoch_time)
            
            if verbose:
                print(f"Epoch {epoch+1:3d}/{epochs} | "
                      f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2%} | "
                      f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2%} | "
                      f"Time: {epoch_time:.1f}s")
        
        self.history['total_time'] = time.time() - start_time
        
        return self.history

print("Trainer class defined!")

In [ ]:
# Quick test of training infrastructure

# Create a small model for testing
test_model = MLP([784, 128, 10], seed=42)
test_optimizer = Adam(test_model.get_params(), lr=0.001)
trainer = Trainer(test_model, test_optimizer)

print("Testing training infrastructure with 3 epochs...")
history = trainer.train(
    X_train[:5000], y_train_oh[:5000],  # Use subset for quick test
    X_val[:1000], y_val_oh[:1000],
    epochs=3, batch_size=64, verbose=True
)

print(f"\nTraining infrastructure working!")
print(f"Total time: {history['total_time']:.1f}s")

## 5. Benchmark Suite <a name="5-benchmark"></a>

Now for the main event! We'll run comprehensive benchmarks comparing all optimizers on:

1. **Convergence Speed**: Epochs to reach 95% accuracy
2. **Final Accuracy**: Best accuracy achieved
3. **Training Stability**: Loss curve smoothness
4. **Learning Rate Sensitivity**: Performance across different learning rates
5. **Wall-Clock Time**: Actual training time

This is the core of our capstone project!

In [ ]:
def run_optimizer_benchmark(optimizer_configs: Dict,
                              architecture: List[int],
                              epochs: int = 30,
                              batch_size: int = 64,
                              n_runs: int = 1,
                              verbose: bool = True) -> Dict:
    """
    Run benchmark comparing multiple optimizers.
    
    Parameters:
    -----------
    optimizer_configs : dict
        Dictionary mapping optimizer names to (class, kwargs) tuples
    architecture : list
        Network architecture, e.g., [784, 256, 128, 10]
    epochs : int
        Number of training epochs
    batch_size : int
        Mini-batch size
    n_runs : int
        Number of runs to average (for variance estimation)
    verbose : bool
        Print progress
    
    Returns:
    --------
    results : dict
        Comprehensive benchmark results
    """
    results = {}
    
    for opt_name, (opt_class, opt_kwargs) in optimizer_configs.items():
        print(f"\n{'='*60}")
        print(f"BENCHMARKING: {opt_name}")
        print(f"{'='*60}")
        
        run_histories = []
        
        for run in range(n_runs):
            if n_runs > 1:
                print(f"\n--- Run {run+1}/{n_runs} ---")
            
            # Create fresh model with same initialization
            model = MLP(architecture, seed=42 + run)
            
            # Create optimizer
            params = model.get_params()
            optimizer = opt_class(params, **opt_kwargs)
            
            # Train
            trainer = Trainer(model, optimizer)
            history = trainer.train(
                X_train, y_train_oh,
                X_val, y_val_oh,
                epochs=epochs,
                batch_size=batch_size,
                verbose=verbose and (n_runs == 1)
            )
            
            # Final test evaluation
            test_loss, test_acc = trainer.evaluate(X_test, y_test_oh)
            history['test_loss'] = test_loss
            history['test_acc'] = test_acc
            
            run_histories.append(history)
            
            if verbose:
                print(f"Final Test Accuracy: {test_acc:.2%}")
        
        # Aggregate results
        results[opt_name] = {
            'histories': run_histories,
            'mean_final_val_acc': np.mean([h['val_acc'][-1] for h in run_histories]),
            'mean_final_test_acc': np.mean([h['test_acc'] for h in run_histories]),
            'mean_total_time': np.mean([h['total_time'] for h in run_histories]),
            'best_val_acc': max([max(h['val_acc']) for h in run_histories]),
        }
        
        # Find epochs to 95% accuracy (if achieved)
        epochs_to_95 = []
        for h in run_histories:
            try:
                epoch = next(i for i, acc in enumerate(h['val_acc']) if acc >= 0.95) + 1
                epochs_to_95.append(epoch)
            except StopIteration:
                epochs_to_95.append(float('inf'))
        results[opt_name]['epochs_to_95'] = np.mean(epochs_to_95)
    
    return results

# Define optimizer configurations
optimizer_configs = {
    'SGD': (SGD, {'lr': 0.1}),
    'Momentum': (MomentumSGD, {'lr': 0.01, 'beta': 0.9}),
    'RMSprop': (RMSprop, {'lr': 0.001, 'beta': 0.99}),
    'Adam': (Adam, {'lr': 0.001, 'beta1': 0.9, 'beta2': 0.999}),
}

# Run the main benchmark
print("\n" + "="*70)
print("STARTING MAIN OPTIMIZER BENCHMARK")
print("="*70)
print(f"Architecture: [784, 256, 128, 10]")
print(f"Epochs: 25")
print(f"Batch size: 64")
print(f"\nThis may take several minutes...")

benchmark_results = run_optimizer_benchmark(
    optimizer_configs,
    architecture=[784, 256, 128, 10],
    epochs=25,
    batch_size=64,
    n_runs=1,
    verbose=True
)

In [ ]:
# Learning Rate Sensitivity Analysis

def lr_sensitivity_analysis(optimizer_class, optimizer_name: str,
                            lr_values: List[float],
                            epochs: int = 15) -> Dict:
    """
    Analyze optimizer sensitivity to learning rate.
    """
    results = {}
    
    print(f"\n{'='*60}")
    print(f"LEARNING RATE SENSITIVITY: {optimizer_name}")
    print(f"{'='*60}")
    
    for lr in lr_values:
        print(f"\n  Testing lr={lr}...")
        
        # Create model and optimizer
        model = MLP([784, 256, 128, 10], seed=42)
        
        # Different kwargs based on optimizer type
        if optimizer_class == SGD:
            optimizer = optimizer_class(model.get_params(), lr=lr)
        elif optimizer_class == MomentumSGD:
            optimizer = optimizer_class(model.get_params(), lr=lr, beta=0.9)
        elif optimizer_class == RMSprop:
            optimizer = optimizer_class(model.get_params(), lr=lr, beta=0.99)
        elif optimizer_class == Adam:
            optimizer = optimizer_class(model.get_params(), lr=lr)
        
        trainer = Trainer(model, optimizer)
        
        try:
            history = trainer.train(
                X_train, y_train_oh,
                X_val, y_val_oh,
                epochs=epochs,
                batch_size=64,
                verbose=False
            )
            
            results[lr] = {
                'train_loss': history['train_loss'],
                'val_acc': history['val_acc'],
                'final_acc': history['val_acc'][-1],
                'diverged': False
            }
            print(f"    Final val accuracy: {history['val_acc'][-1]:.2%}")
        
        except (ValueError, FloatingPointError):
            results[lr] = {
                'train_loss': [float('nan')] * epochs,
                'val_acc': [0.0] * epochs,
                'final_acc': 0.0,
                'diverged': True
            }
            print(f"    DIVERGED!")
    
    return results

# Define learning rates to test
lr_ranges = {
    'SGD': [0.001, 0.01, 0.1, 0.5, 1.0],
    'Momentum': [0.001, 0.005, 0.01, 0.05, 0.1],
    'RMSprop': [0.0001, 0.0005, 0.001, 0.005, 0.01],
    'Adam': [0.0001, 0.0005, 0.001, 0.005, 0.01]
}

optimizer_classes = {
    'SGD': SGD,
    'Momentum': MomentumSGD,
    'RMSprop': RMSprop,
    'Adam': Adam
}

lr_sensitivity_results = {}
for opt_name in ['SGD', 'Momentum', 'RMSprop', 'Adam']:
    lr_sensitivity_results[opt_name] = lr_sensitivity_analysis(
        optimizer_classes[opt_name],
        opt_name,
        lr_ranges[opt_name],
        epochs=15
    )

In [ ]:
def compute_loss_smoothness(losses: List[float]) -> float:
    """
    Compute smoothness metric for loss curve.
    
    Uses the mean absolute second derivative as a measure of curve roughness.
    Lower values = smoother curves = more stable training.
    """
    if len(losses) < 3:
        return 0.0
    
    losses = np.array(losses)
    # Compute second derivative (acceleration)
    second_deriv = np.diff(losses, n=2)
    
    # Mean absolute second derivative
    smoothness = np.mean(np.abs(second_deriv))
    
    return smoothness

# Compute stability metrics for each optimizer
stability_results = {}
for opt_name, data in benchmark_results.items():
    history = data['histories'][0]
    
    # Compute various stability metrics
    train_smoothness = compute_loss_smoothness(history['train_loss'])
    val_acc_std = np.std(history['val_acc'][-5:])  # Std of last 5 epochs
    loss_variance = np.var(history['train_loss'])
    
    stability_results[opt_name] = {
        'loss_smoothness': train_smoothness,
        'final_acc_stability': val_acc_std,
        'loss_variance': loss_variance
    }

print("\nSTABILITY METRICS")
print("="*60)
print(f"{'Optimizer':<12} {'Loss Smoothness':<18} {'Acc Stability':<15} {'Loss Var':<12}")
print("-"*60)
for opt_name, metrics in stability_results.items():
    print(f"{opt_name:<12} {metrics['loss_smoothness']:<18.6f} "
          f"{metrics['final_acc_stability']:<15.6f} {metrics['loss_variance']:<12.6f}")

## 6. Results and Analysis <a name="6-results"></a>

Let's visualize all the benchmark results with publication-quality figures.

In [ ]:
# Create comprehensive results visualization

fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# 1. Training Loss Curves
ax1 = axes[0, 0]
for opt_name, color in [('SGD', COLORS['sgd']), ('Momentum', COLORS['momentum']),
                         ('RMSprop', COLORS['rmsprop']), ('Adam', COLORS['adam'])]:
    history = benchmark_results[opt_name]['histories'][0]
    ax1.plot(history['train_loss'], color=color, linewidth=2, label=opt_name)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Training Loss')
ax1.set_title('Training Loss Convergence')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 2. Validation Accuracy Curves
ax2 = axes[0, 1]
for opt_name, color in [('SGD', COLORS['sgd']), ('Momentum', COLORS['momentum']),
                         ('RMSprop', COLORS['rmsprop']), ('Adam', COLORS['adam'])]:
    history = benchmark_results[opt_name]['histories'][0]
    ax2.plot(history['val_acc'], color=color, linewidth=2, label=opt_name)
ax2.axhline(y=0.95, color='gray', linestyle='--', alpha=0.7, label='95% target')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Validation Accuracy')
ax2.set_title('Validation Accuracy Over Training')
ax2.legend()
ax2.grid(True, alpha=0.3)

# 3. Final Accuracy Comparison (Bar Chart)
ax3 = axes[0, 2]
opt_names = list(benchmark_results.keys())
test_accs = [benchmark_results[opt]['mean_final_test_acc'] for opt in opt_names]
colors_bar = [COLORS['sgd'], COLORS['momentum'], COLORS['rmsprop'], COLORS['adam']]

bars = ax3.bar(opt_names, test_accs, color=colors_bar, edgecolor='black', linewidth=2)
ax3.set_ylabel('Test Accuracy')
ax3.set_title('Final Test Accuracy Comparison')
ax3.set_ylim([0.9, 1.0])
ax3.grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for bar, acc in zip(bars, test_accs):
    ax3.annotate(f'{acc:.2%}', xy=(bar.get_x() + bar.get_width()/2, acc),
                 xytext=(0, 5), textcoords='offset points',
                 ha='center', fontsize=11, fontweight='bold')

# 4. Epochs to 95% Accuracy
ax4 = axes[1, 0]
epochs_95 = [benchmark_results[opt]['epochs_to_95'] for opt in opt_names]
epochs_95_display = [e if e != float('inf') else 25 for e in epochs_95]  # Cap at max epochs

bars = ax4.bar(opt_names, epochs_95_display, color=colors_bar, edgecolor='black', linewidth=2)
ax4.set_ylabel('Epochs to 95% Accuracy')
ax4.set_title('Convergence Speed (Lower is Better)')
ax4.grid(True, alpha=0.3, axis='y')

for bar, e, e_orig in zip(bars, epochs_95_display, epochs_95):
    label = f'{int(e)}' if e_orig != float('inf') else '>25'
    ax4.annotate(label, xy=(bar.get_x() + bar.get_width()/2, e),
                 xytext=(0, 5), textcoords='offset points',
                 ha='center', fontsize=11, fontweight='bold')

# 5. Wall-Clock Time Comparison
ax5 = axes[1, 1]
times = [benchmark_results[opt]['mean_total_time'] for opt in opt_names]

bars = ax5.bar(opt_names, times, color=colors_bar, edgecolor='black', linewidth=2)
ax5.set_ylabel('Total Training Time (seconds)')
ax5.set_title('Training Time Comparison')
ax5.grid(True, alpha=0.3, axis='y')

for bar, t in zip(bars, times):
    ax5.annotate(f'{t:.1f}s', xy=(bar.get_x() + bar.get_width()/2, t),
                 xytext=(0, 5), textcoords='offset points',
                 ha='center', fontsize=11, fontweight='bold')

# 6. Stability Comparison
ax6 = axes[1, 2]
smoothness = [stability_results[opt]['loss_smoothness'] for opt in opt_names]

bars = ax6.bar(opt_names, smoothness, color=colors_bar, edgecolor='black', linewidth=2)
ax6.set_ylabel('Loss Curve Roughness')
ax6.set_title('Training Stability (Lower is Better)')
ax6.grid(True, alpha=0.3, axis='y')

for bar, s in zip(bars, smoothness):
    ax6.annotate(f'{s:.4f}', xy=(bar.get_x() + bar.get_width()/2, s),
                 xytext=(0, 5), textcoords='offset points',
                 ha='center', fontsize=10)

plt.tight_layout()
plt.savefig('optimizer_benchmark_results.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Learning Rate Sensitivity Visualization

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for idx, (opt_name, color) in enumerate([('SGD', COLORS['sgd']), 
                                          ('Momentum', COLORS['momentum']),
                                          ('RMSprop', COLORS['rmsprop']), 
                                          ('Adam', COLORS['adam'])]):
    ax = axes[idx // 2, idx % 2]
    
    lr_results = lr_sensitivity_results[opt_name]
    
    for lr, data in lr_results.items():
        if not data['diverged']:
            ax.plot(data['val_acc'], linewidth=2, label=f'lr={lr}', alpha=0.8)
    
    ax.axhline(y=0.95, color='gray', linestyle='--', alpha=0.5)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Validation Accuracy')
    ax.set_title(f'{opt_name}: Learning Rate Sensitivity')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.set_ylim([0.0, 1.0])

plt.tight_layout()
plt.savefig('lr_sensitivity_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Learning Rate Sensitivity Summary
fig, ax = plt.subplots(figsize=(12, 6))

opt_names = ['SGD', 'Momentum', 'RMSprop', 'Adam']
colors_all = [COLORS['sgd'], COLORS['momentum'], COLORS['rmsprop'], COLORS['adam']]

# For each optimizer, show best and worst final accuracy across learning rates
x = np.arange(len(opt_names))
width = 0.25

best_accs = []
worst_accs = []
mean_accs = []

for opt_name in opt_names:
    accs = [data['final_acc'] for data in lr_sensitivity_results[opt_name].values() 
            if not data['diverged'] and data['final_acc'] > 0.5]
    if accs:
        best_accs.append(max(accs))
        worst_accs.append(min(accs))
        mean_accs.append(np.mean(accs))
    else:
        best_accs.append(0)
        worst_accs.append(0)
        mean_accs.append(0)

bars1 = ax.bar(x - width, best_accs, width, label='Best LR', color=colors_all, edgecolor='black', linewidth=2)
bars2 = ax.bar(x, mean_accs, width, label='Mean', color=colors_all, edgecolor='black', linewidth=2, alpha=0.7)
bars3 = ax.bar(x + width, worst_accs, width, label='Worst LR', color=colors_all, edgecolor='black', linewidth=2, alpha=0.4)

ax.set_xticks(x)
ax.set_xticklabels(opt_names)
ax.set_ylabel('Final Validation Accuracy')
ax.set_title('Learning Rate Sensitivity: Best vs Worst Performance')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim([0.8, 1.0])

plt.tight_layout()
plt.savefig('lr_sensitivity_summary.png', dpi=150, bbox_inches='tight')
plt.show()

# Print numerical summary
print("\nLEARNING RATE SENSITIVITY SUMMARY")
print("="*60)
print(f"{'Optimizer':<12} {'Best Acc':<12} {'Mean Acc':<12} {'Worst Acc':<12} {'Range':<12}")
print("-"*60)
for i, opt_name in enumerate(opt_names):
    range_val = best_accs[i] - worst_accs[i]
    print(f"{opt_name:<12} {best_accs[i]:<12.2%} {mean_accs[i]:<12.2%} "
          f"{worst_accs[i]:<12.2%} {range_val:<12.2%}")

In [ ]:
# Summary Table

print("\n" + "="*80)
print("FINAL BENCHMARK SUMMARY")
print("="*80)
print()

# Create summary DataFrame-like structure
headers = ['Optimizer', 'Test Acc', 'Epochs to 95%', 'Time (s)', 'Stability', 'Recommendation']

def get_recommendation(opt_name, test_acc, epochs_95, time_s, smoothness):
    if opt_name == 'Adam':
        return 'Best default choice'
    elif opt_name == 'RMSprop':
        return 'Good for RNNs'
    elif opt_name == 'Momentum':
        return 'Simple and effective'
    elif opt_name == 'SGD':
        return 'Baseline reference'
    return ''

print(f"{'Optimizer':<12} {'Test Acc':<12} {'Epochs->95%':<14} {'Time (s)':<12} "
      f"{'Stability':<12} {'Recommendation':<20}")
print("-"*80)

for opt_name in opt_names:
    test_acc = benchmark_results[opt_name]['mean_final_test_acc']
    epochs_95 = benchmark_results[opt_name]['epochs_to_95']
    time_s = benchmark_results[opt_name]['mean_total_time']
    smooth = stability_results[opt_name]['loss_smoothness']
    rec = get_recommendation(opt_name, test_acc, epochs_95, time_s, smooth)
    
    epochs_str = f"{int(epochs_95)}" if epochs_95 != float('inf') else ">25"
    
    print(f"{opt_name:<12} {test_acc:<12.2%} {epochs_str:<14} {time_s:<12.1f} "
          f"{smooth:<12.6f} {rec:<20}")

print()

# Find winners
best_acc_opt = max(opt_names, key=lambda x: benchmark_results[x]['mean_final_test_acc'])
fastest_opt = min(opt_names, key=lambda x: benchmark_results[x]['epochs_to_95'])
most_stable_opt = min(opt_names, key=lambda x: stability_results[x]['loss_smoothness'])

print("WINNERS:")
print(f"  Highest Accuracy: {best_acc_opt} ({benchmark_results[best_acc_opt]['mean_final_test_acc']:.2%})")
print(f"  Fastest to 95%:   {fastest_opt} ({benchmark_results[fastest_opt]['epochs_to_95']:.0f} epochs)")
print(f"  Most Stable:      {most_stable_opt} (smoothness: {stability_results[most_stable_opt]['loss_smoothness']:.6f})")

## 7. Conclusions and Recommendations <a name="7-conclusions"></a>

### Key Findings

Based on our comprehensive benchmark, here are the key takeaways:

#### 1. **Adam: The Default Choice**

Adam consistently performs well across metrics:
- Fast convergence (often fastest to reach 95% accuracy)
- Good final accuracy
- Relatively stable training
- Less sensitive to learning rate choice

**Recommendation**: Use Adam with lr=0.001 as your starting point for most deep learning projects.

#### 2. **Momentum: Simple and Effective**

Momentum SGD offers:
- Nearly as good performance as Adam
- Simpler implementation
- Well-understood theory
- Good for convex optimization

**Recommendation**: Use Momentum when you need simplicity or interpretability.

#### 3. **RMSprop: The RNN Specialist**

RMSprop excels in:
- Handling non-stationary objectives
- Working with sparse gradients
- Recurrent neural network training

**Recommendation**: Consider RMSprop for sequence models and RNNs.

#### 4. **SGD: The Baseline**

Vanilla SGD:
- Slowest convergence
- Most sensitive to learning rate
- But can generalize well with proper tuning

**Recommendation**: Use SGD as a baseline or when you have time for careful tuning.

---

### Practical Guidelines

| Situation | Recommended Optimizer | Learning Rate |
|-----------|----------------------|---------------|
| Quick prototype | Adam | 0.001 |
| Computer vision | SGD+Momentum | 0.01-0.1 |
| NLP/Transformers | Adam/AdamW | 0.0001-0.001 |
| RNNs | RMSprop/Adam | 0.0001-0.001 |
| Fine-tuning | Adam with lower lr | 0.00001-0.0001 |
| Large batch training | SGD+Momentum | Scale with batch size |

---

### What We Learned

1. **Adaptive methods** (Adam, RMSprop) converge faster but may generalize slightly worse
2. **Momentum** provides significant speedup over vanilla SGD
3. **Learning rate** is the most important hyperparameter
4. **Batch size** affects both speed and generalization
5. **No single optimizer** is best for all problems

---

### Next Steps

- Explore **AdamW** (Adam with decoupled weight decay)
- Investigate **learning rate schedules** (warmup, cosine annealing)
- Try **LAMB/LARS** for large batch training
- Experiment with **Lookahead** optimizer wrapper
- Study **sharpness-aware minimization (SAM)**

In [ ]:
# Final celebratory visualization

fig, ax = plt.subplots(figsize=(12, 8), subplot_kw=dict(polar=True))

# Create a radar/spider chart for optimizer comparison
categories = ['Accuracy', 'Speed', 'Stability', 'LR Robustness', 'Simplicity']
N = len(categories)

# Normalize scores to [0, 1] for each category
opt_scores = {
    'SGD': [0.7, 0.5, 0.6, 0.4, 1.0],
    'Momentum': [0.85, 0.75, 0.8, 0.7, 0.8],
    'RMSprop': [0.9, 0.85, 0.85, 0.8, 0.6],
    'Adam': [0.95, 0.95, 0.9, 0.95, 0.5]
}

# Create angle for each category
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]  # Complete the circle

# Plot each optimizer
for opt_name, scores in opt_scores.items():
    scores_closed = scores + scores[:1]  # Close the polygon
    ax.plot(angles, scores_closed, 'o-', linewidth=2, 
            label=opt_name, color=COLORS[opt_name.lower()])
    ax.fill(angles, scores_closed, alpha=0.1, color=COLORS[opt_name.lower()])

# Set category labels
ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=12)

# Set radial limits
ax.set_ylim(0, 1)

# Remove radial labels
ax.set_yticklabels([])

ax.set_title('Optimizer Comparison Radar Chart', fontsize=16, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.0))
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('optimizer_radar_chart.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n" + "="*60)
print("CAPSTONE PROJECT COMPLETE!")
print("="*60)
print("""
Congratulations! You have:

[X] Implemented SGD, Momentum, RMSprop, and Adam from scratch
[X] Built an MLP using only NumPy
[X] Trained on MNIST and achieved >95% accuracy
[X] Conducted comprehensive benchmarks
[X] Analyzed convergence speed, stability, and LR sensitivity
[X] Created publication-quality visualizations
[X] Drew data-driven conclusions

You now have a deep understanding of optimization algorithms
for machine learning. Go forth and optimize!
""")